# 02 — House PDF download & manifest

## Objectif

Télécharger les PDF PTR House listés dans `house_ptr_index.csv`.

Output principal : `data/audit/house_pdf_manifest.csv`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.house_download import download_all_pdfs, summarize_manifest, write_download_report
from src.utils import PROCESSED_HOUSE_DIR, AUDIT_DIR, RAW_HOUSE_PDF_DIR, require_file

print("Imports OK")

In [ ]:
SLEEP_SECONDS = 0.25
FORCE_DOWNLOAD = False
MAX_FILES = None  # Mettre 20 pour un test rapide avant le run complet.
SAVE_EVERY = 100

ptr_index_path = PROCESSED_HOUSE_DIR / "house_ptr_index.csv"
manifest_path = AUDIT_DIR / "house_pdf_manifest.csv"

In [ ]:
require_file(ptr_index_path, "Run 01_house_index_audit.ipynb first.")
df_ptr = pd.read_csv(ptr_index_path, dtype={"doc_id": str})

print("df_ptr shape:", df_ptr.shape)
print("years:", sorted(df_ptr["year"].unique()))
display(df_ptr.head())

## Téléchargement

Le downloader est idempotent : un PDF existant et valide est conservé.

In [ ]:
manifest = download_all_pdfs(
    df_ptr,
    output_base_dir=RAW_HOUSE_PDF_DIR,
    manifest_path=manifest_path,
    sleep_seconds=SLEEP_SECONDS,
    force=FORCE_DOWNLOAD,
    max_files=MAX_FILES,
    save_every=SAVE_EVERY,
)

print("manifest shape:", manifest.shape)
display(manifest.head())

In [ ]:
summary = summarize_manifest(manifest, expected_count=len(df_ptr) if MAX_FILES is None else MAX_FILES)
summary

In [ ]:
if not manifest.empty:
    display(manifest["download_status"].value_counts(dropna=False))
    problems = manifest[~manifest["download_status"].isin(["ok", "skipped_existing"])]
    display(problems.head(20))

In [ ]:
report_path = write_download_report(summary, manifest_path=manifest_path)
print("Written:", manifest_path.relative_to(ROOT))
print("Written:", report_path.relative_to(ROOT))

## Conclusion

**OK** si le manifest explique les succès, skips et échecs.

**NEXT** — auditer la qualité texte avec `03_house_pdf_quality_smoke_test.ipynb`.